In [1]:
# 11_utility_validation.ipynb -- Validate the utility-proxy ORDERING against a measured, variance-bearing signal.
# The utility proxy (nb 03/06) is a deterministic policy index with no file-level variance; a reviewer may ask
# whether its ordering tracks anything real. Here we measure, per file and tier, a STRUCTURAL-TOKEN RETENTION rate:
# the fraction of the original file's structure-bearing tokens (Java keywords, control-flow constructs, operators,
# bracketing) that survive masking verbatim -- the control- and type-structure a model must see to produce a correct
# completion, as distinct from identifier NAMES. We then check (a) that this measured signal reproduces the proxy's
# ordering across tiers (Spearman), and (b) that the key ordinal claim C2 > C3 holds under a paired per-file test.
# No LLM is needed; reuses data/masked_corpus.json from notebook 02. Output paths are not printed (anonymity).

import json, re
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

with open(DATA / "masked_corpus.json", encoding="utf-8") as f:
    masked = json.load(f)
TIERS = ["C0", "C1", "C2", "C3", "C4"]

# --- Paper's stipulated utility proxy (from nb 03), for comparison -----------
tot_id = sum(r["identifier"] for r in masked["C2"]) or 1
tot_lit = sum(r["literal"] for r in masked["C3"]) or 1
W_FLOOR, W_U_ID, W_U_LIT = 0.30, 0.45, 0.25
def proxy_utility(tier):
    id_m = sum(r["identifier"] for r in masked[tier]) / tot_id
    lit_m = sum(r["literal"] for r in masked[tier]) / tot_lit
    return W_FLOOR + W_U_ID * (1 - id_m) + W_U_LIT * (1 - lit_m)

# --- Structure-bearing tokens: keywords/control-flow + operators + brackets --
STRUCT_KW = set("""if else for while do switch case break continue return try catch finally throw throws
new class interface enum extends implements public private protected static final abstract void int long
double float boolean char byte short this super instanceof synchronized import package""".split())
OP = re.compile(r"==|!=|<=|>=|\+\+|--|->|::|&&|\|\||[{}()\[\];,.<>+\-*/%=&|!?:]")
WORD = re.compile(r"[A-Za-z_$][A-Za-z0-9_$]*")

def struct_tokens(code):
    return [w for w in WORD.findall(code) if w in STRUCT_KW] + OP.findall(code)

def multiset_retention(orig, msk):
    from collections import Counter
    co, cm = Counter(struct_tokens(orig)), Counter(struct_tokens(msk))
    if not co:
        return np.nan
    return sum(min(co[t], cm[t]) for t in co) / sum(co.values())

def boot_ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    if len(a) == 0:
        return (np.nan, np.nan)
    m = a[rng.integers(0, len(a), size=(10000, len(a)))].mean(axis=1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

# --- Per-file, per-tier structural retention --------------------------------
orig_by_file = {r["file_id"]: r["code"] for r in masked["C0"]}
per_tier = {}
for tier in TIERS:
    vals = [multiset_retention(orig_by_file[r["file_id"]], r["code"]) for r in masked[tier]]
    per_tier[tier] = np.array(vals, float)

rows = []
for tier in TIERS:
    sr = per_tier[tier]
    lo, hi = boot_ci(sr)
    rows.append({"tier": tier, "proxy_utility": proxy_utility(tier),
                 "struct_retention": float(np.nanmean(sr)), "sr_ci_lo": lo, "sr_ci_hi": hi})
df = pd.DataFrame(rows)
df.to_csv(TAB / "utility_validation.csv", index=False)

# --- (a) Ordinal agreement: Spearman(proxy, measured) across tiers ----------
try:
    from scipy import stats as st
    rho, pval = st.spearmanr(df["proxy_utility"], df["struct_retention"])
except Exception:
    rho, pval = float("nan"), float("nan")

# --- (b) Paired per-file test of the key claim C2 > C3 ----------------------
c2, c3 = per_tier["C2"], per_tier["C3"]
mask = ~(np.isnan(c2) | np.isnan(c3))
diff = c2[mask] - c3[mask]
dz = diff.mean() / (diff.std(ddof=1) + 1e-12)
dlo, dhi = boot_ci(diff)
try:
    from scipy import stats as st
    wp = st.wilcoxon(c2[mask], c3[mask], zero_method="wilcox").pvalue if np.any(diff != 0) else 1.0
except Exception:
    wp = float("nan")

# --- Report -----------------------------------------------------------------
print("Utility-proxy validation (structural-token retention; N = %d files):" % int(mask.sum()))
print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\n(a) Ordinal agreement across tiers: Spearman rho = {rho:.3f} (p = {pval:.3f})")
print(f"(b) C2 > C3 (paired, measured retention): mean diff = {diff.mean():.3f} "
      f"95% CI [{dlo:.3f}, {dhi:.3f}], d_z = {dz:.2f}, Wilcoxon p = {wp:.2e}")
print("\nReading: the measured signal reproduces the proxy ORDERING (high rho) and confirms C2 retains more")
print("structure than C3. The measured GAP (C2-C3) is small in absolute terms, so the proxy is read ordinally;")
print("that C3 still retains ~94-95% of structure quantifies the logic-exposure channel masking does not close.")


Utility-proxy validation (structural-token retention; N = 150 files):
tier  proxy_utility  struct_retention  sr_ci_lo  sr_ci_hi
  C0          1.000             1.000     1.000     1.000
  C1          1.000             0.968     0.965     0.970
  C2          0.550             0.968     0.965     0.970
  C3          0.300             0.945     0.937     0.952
  C4          0.300             0.945     0.937     0.952

(a) Ordinal agreement across tiers: Spearman rho = 0.973 (p = 0.005)
(b) C2 > C3 (paired, measured retention): mean diff = 0.022 95% CI [0.016, 0.031], d_z = 0.48, Wilcoxon p = 2.29e-26

Reading: the measured signal reproduces the proxy ORDERING (high rho) and confirms C2 retains more
structure than C3. The measured GAP (C2-C3) is small in absolute terms, so the proxy is read ordinally;
that C3 still retains ~94-95% of structure quantifies the logic-exposure channel masking does not close.
